# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import files
uploaded = files.upload()  # select capstone_features.csv
import pandas as pd
df = pd.read_csv('capstone_features.csv')
feats = ['imp_prev30','visible_queries','rare_share','anon_share','top_query_share','pos_volatility_60d']
print(f'{len(df):,} rows loaded')

Saving capstone_features.csv to capstone_features (1).csv
111,247 rows loaded


## 1. Build the feature vector

Six features, all numeric: imp_prev30, visible_queries, rare_share,
anon_share, top_query_share, pos_volatility_60d.

In [7]:
print(df[feats].head())
print(df[feats].dtypes)


   imp_prev30  visible_queries  rare_share  anon_share  top_query_share  \
0       955.0              1.0    0.022750    0.957216         1.000000   
1      3338.0             14.0    0.017946    0.932994         0.181818   
2       130.0              3.0    0.162037    0.552469         0.827027   
3       340.0              2.0    0.108932    0.820261         0.800000   
4       531.0              5.0    0.163052    0.788332         0.215385   

   pos_volatility_60d  
0            3.428418  
1            1.812187  
2           13.138960  
3           14.892146  
4            9.873715  
imp_prev30            float64
visible_queries       float64
rare_share            float64
anon_share            float64
top_query_share       float64
pos_volatility_60d    float64
dtype: object


## 2. Feature notes (meaning, missing, categorical, available-when?)

- imp_prev30: prior-30-day impressions. Never missing (used as a HAVING >=100 filter upstream).
- visible_queries, rare_share, anon_share, top_query_share: from the query-mix
  table, missing for 9,044 rows (~8%) — pages with no matching query-level
  data in the 90-day window. Numeric, continuous.
- pos_volatility_60d: std. dev. of position over 60 days. Missing for 1 row
  (a page with only a single day of data, so no variance is computable).
- Available-when: all six are computed from the PRIOR 30/60-day window,
  before the last-30-day window used to define the label — so all are
  legitimately available at prediction time, not after the fact.

In [8]:
print(df[feats].isna().sum())

imp_prev30               0
visible_queries       9044
rare_share            9044
anon_share            9044
top_query_share       9044
pos_volatility_60d       1
dtype: int64


## 3. The leakage hunt

Checked for: (1) any feature computed from the same days as the label window
— none are, features use prior-30/60 days, label uses last-30 days.
(2) client_hash_id/content_hash_id leaking into the model — they don't;
used only for joining and for the GroupShuffleSplit, never as features.
(3) any pre-built trend/direction column from the warehouse — none used;
is_declining was built from raw impression counts, not borrowed.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

Excluded: client_hash_id, content_hash_id (identifiers, not signal, and
including them would let the model memorize specific clients rather than
learn generalizable patterns — confirmed by the accuracy drop under a
client-grouped split vs. a random split). Excluded any trend_direction/
trend_pct-style precomputed label field, to avoid inheriting someone else's
label logic or an unknown leakage path.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.